In [1]:
import random
import string

In [2]:
CHARS = list(string.ascii_lowercase + ' ')
CHAR_TO_INDEX = {c: i for i, c in enumerate(CHARS)}
INDEX_TO_CHAR = {i: c for c, i in CHAR_TO_INDEX.items()}

INDEX_TO_CHAR

{0: 'a',
 1: 'b',
 2: 'c',
 3: 'd',
 4: 'e',
 5: 'f',
 6: 'g',
 7: 'h',
 8: 'i',
 9: 'j',
 10: 'k',
 11: 'l',
 12: 'm',
 13: 'n',
 14: 'o',
 15: 'p',
 16: 'q',
 17: 'r',
 18: 's',
 19: 't',
 20: 'u',
 21: 'v',
 22: 'w',
 23: 'x',
 24: 'y',
 25: 'z',
 26: ' '}

In [3]:
VOCAB_SIZE = len(CHARS)

VOCAB_SIZE

27

In [4]:
def caesar_encrypt(text, k):
    res = ''
    for c in text:
        if c in CHAR_TO_INDEX:
            idx = (CHAR_TO_INDEX[c] + k) % VOCAB_SIZE
            res += INDEX_TO_CHAR[idx]
        else:
            res += c
    return res

def generate_data(n=10000, max_len=20, k=3):
    X, Y = [], []
    for _ in range(n):
        length = random.randint(5, max_len)
        text = ''.join(random.choice(CHARS) for _ in range(length))
        enc = caesar_encrypt(text, k)
        X.append(enc)
        Y.append(text)
    return X, Y

In [5]:
import torch

In [6]:
MAX_LEN = 20

def encode(texts):
    X = torch.zeros((len(texts), MAX_LEN), dtype=torch.long)
    for i, t in enumerate(texts):
        for j, c in enumerate(t[:MAX_LEN]):
            X[i, j] = CHAR_TO_INDEX[c]
    return X

X_text, Y_text = generate_data()

X = encode(X_text)
Y = encode(Y_text)

X_text

['qrcd hkuomhrzfwwe',
 'wozgpsnv xawahesqaxi',
 'tlcxletqdt',
 'ysvxsolrmxhpxrr bc',
 'ddxypvidyu',
 'heypuvaqtjjkoudnd',
 'pu fn wmxdvswccl',
 'ccxnm',
 'weiv xpi jzmhfffnh',
 'oxjjbhda',
 'vkhgzp',
 'pnnhqbnlckhfs vti',
 'lzjxuzuxxn',
 'cidthebg so',
 'ggitfsfjfkcqiuz',
 'avvfigwsllf jfajnl',
 'qhmwebmjfokm',
 'jwupviq yrmcvzcf',
 'czwaasbjftaksoz',
 'ucfokdxn vgxkobngq',
 'wqcaebehmxl',
 ' qhesa',
 'ihnvpjdaquzdca',
 'exiitairkr',
 'xbigaiajb ',
 'jzeqhxmoxep xq',
 'plnwljbd',
 'unifxclggavvke',
 'wsjdtfoollv',
 'nqwwxugfxoresdfjwvxk',
 'uvwgrkjm',
 'inzfja aurr',
 'jhmmiswhdzeyndgwpamx',
 'ljsxowfnchmc  vtr',
 'dxknzab',
 'na vyytd',
 '  fyxurrrbvowg',
 'tkibyvsnmcr',
 'blcblcfszod',
 'pahfjppl',
 'mcitpmnfw',
 'eriigtp cp',
 'ilebdxj htrhju',
 'mmmsjxeevkdoljf',
 'iguabksyy',
 'ttihdkociyvevtbqmfvu',
 'brtlrnplnzpggfdfydlz',
 'ypxcpbtvpti',
 'uzrwglwfylnuf yhtxh ',
 'quaqxueakzd',
 'frcmdroi',
 'orholvho',
 'ozzobsmswdvws',
 'qdlmjzpbduzoarlrmoys',
 'cqieh',
 'ehlbgzdkusjkpukn',
 

In [7]:
class CaesarNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = torch.nn.Embedding(VOCAB_SIZE, 32)
        self.rnn = torch.nn.RNN(32, 64, batch_first=True)
        self.fc = torch.nn.Linear(64, VOCAB_SIZE)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.rnn(x)
        return self.fc(x)

In [8]:
model = CaesarNet()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for ep in range(10):
    total_loss = 0
    for i in range(0, len(X), 64):
        xb = X[i:i+64]
        yb = Y[i:i+64]

        pred = model(xb)
        pred = pred.view(-1, VOCAB_SIZE)
        yb = yb.view(-1)

        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'epoch {ep}, loss {total_loss:.4f}')

epoch 0, loss 27.8584
epoch 1, loss 6.2817
epoch 2, loss 5.8627
epoch 3, loss 5.6777
epoch 4, loss 5.6243
epoch 5, loss 5.5566
epoch 6, loss 5.5353
epoch 7, loss 5.4513
epoch 8, loss 5.4515
epoch 9, loss 5.4900


In [9]:
def decode(tensor):
    return ''.join(INDEX_TO_CHAR[i.item()] for i in tensor)

def test(text):
    enc = caesar_encrypt(text, 3)
    x = encode([enc])
    pred = model(x)[0].argmax(dim=1)
    decoded = decode(pred)
    return enc, decoded[:len(text)]

print(test('hello world'))

('khoorczruog', 'hello world')
